In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
import json
import re 
import math
import concurrent.futures
from pymongo import MongoClient
import os
import json
import os
import json
import time
import urllib.parse
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

MAX_WORKERS = 1

collection_name = "migros_products"
BASE_URL = "https://www.migros.ch/fr/category/"
urls_migros = {
    "FRUITS_LEGUMES": "fruits-legumes",
    "VIANDES_POISSONS": "viandes-poissons",
    "PRODUITS_LAITIERS_OEUFS_PLATS": "produits-laitiers-ufs-plats-prep",
    "BOULANGERIE_PATISSERIE_PETIT_DEJ": "boulangerie-patisserie-petit-dej",
    "PATES_CONDIMENTS_CONSERVES": "pates-condiments-conserves",
    "SNACKS_CONFISERIES": "snacks-confiseries",
    "SURGELES": "surgeles",
    "BOISSONS_CAFE_THE": "boissons-cafe-the",
    "VINS_BIERES_SPIRITUEUX": "vins-bieres-spiritueux",
    "COSMETIQUES_DROGUERIE": "cosmetiques-droguerie",
    "ENTRETIEN_NETTOYAGE": "entretien-nettoyage",
    "BEBE_ENFANTS": "bebe-enfants",
    "MENAGE_HABITAT": "menage-habitat",
    "ANIMAUX": "animaux",
}


In [ ]:
import re

def clean_price_to_float(raw_string):
    """Cleans Swiss price formats (1.-, 16.–, 1.−) into sortable floats."""
    if not raw_string:
        return None
    cleaned = raw_string.strip()
    # Regex: Look for a dot followed by any non-digit chars at the end and replace with .00
    cleaned = re.sub(r'\.\D+$', '.00', cleaned)
    try:
        return float(cleaned)
    except ValueError:
        return cleaned

def parse_product(article):
    """Parses a single product card into a dictionary."""
    product_dict = {}
    
    # ID
    link_tag = article.find('a', {'data-testid': 'product-link'})
    product_dict['id'] = link_tag.get('href').rstrip('/').split('/')[-1] if link_tag else None
    
    # Brand & Name
    brand = article.find('span', class_='name')
    product_dict['brand'] = brand.text.strip() if brand else None
    name = article.find('span', attrs={'data-testid': lambda x: x and x.startswith('product-name')})
    product_dict['name'] = name.text.strip() if name else None
    
    # Current Price
    price_tag = article.find('span', {'data-testid': 'current-price'})
    product_dict['price'] = clean_price_to_float(price_tag.text) if price_tag else None
        
    # --- NOUVEAU : Logique de Promotion ---
    product_dict['is_promo'] = False
    product_dict['promo_rate'] = None
    product_dict['original_price'] = None

    # Chercher le badge de promotion
    promo_badge = article.find('span', class_='badge-promo')
    if promo_badge:
        product_dict['is_promo'] = True
        # Extraire le texte de la promo (ex: "20%")
        promo_text_element = promo_badge.find('span', {'data-testid': 'description'})
        if promo_text_element:
            product_dict['promo_rate'] = promo_text_element.text.strip()
            
    # Chercher le prix original s'il y a une promotion
    original_price_element = article.find('span', {'data-testid': 'original-price'})
    if original_price_element:
        product_dict['original_price'] = clean_price_to_float(original_price_element.text)
    # --------------------------------------

    # Quantity (Multipack Math)
    quantity_tag = article.find('span', {'data-testid': 'default-product-size'})
    if quantity_tag:
        raw_qty = quantity_tag.text.strip()
        multipack_match = re.search(r'(\d+)\s*[xX]\s*(\d+(?:\.\d+)?)\s*([a-zA-Z]+)', raw_qty)
        if multipack_match:
            total = int(multipack_match.group(1)) * float(multipack_match.group(2))
            unit = multipack_match.group(3)
            product_dict['quantity'] = f"{int(total) if total.is_integer() else total}{unit}"
        else:
            product_dict['quantity'] = raw_qty
    else:
        product_dict['quantity'] = None
    
    # Price per Unit & Unit
    product_dict['price_per_unit'], product_dict['unit'] = None, None
    ppu_tag = article.find('span', id=lambda x: x and x.endswith('-price-unit'))
    if ppu_tag:
        raw_ppu = ppu_tag.text.strip() 
        if '/' in raw_ppu:
            ppu_part, unit_part = raw_ppu.split('/', 1)
            product_dict['price_per_unit'] = clean_price_to_float(ppu_part)
            product_dict['unit'] = unit_part.strip() 
        else:
            product_dict['price_per_unit'] = clean_price_to_float(raw_ppu)
    
    # Image URL
    img = article.find('img')
    product_dict['image_url'] = img.get('src') if img else None
    
    return product_dict

In [3]:
def create_driver(headless=True):
    """
    Creates and returns a Chrome WebDriver instance.
    """
    chrome_options = Options() if "Options" in globals() else webdriver.ChromeOptions()

    if headless:
        chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--window-size=1920,1080")

    return webdriver.Chrome(options=chrome_options)


def scrape_single_page(url, driver):
    """
    Parses the current state of the driver's DOM. 
    Does NOT reload the page to preserve dynamically loaded content.
    """
    page_products = []
    
    try:
        # Step A: Grab the HTML currently loaded in the browser
        rendered_html = driver.page_source
        
        if rendered_html:
            # --- DEBUG: Save the FULL expanded HTML to a file ---
            # We use a specific suffix to distinguish it from a single-page load
            safe_filename = urllib.parse.quote_plus(url) + "_FULLY_EXPANDED.html"
            debug_dir = "debug_html"
            os.makedirs(debug_dir, exist_ok=True)
            
            debug_filepath = os.path.join(debug_dir, safe_filename)
            with open(debug_filepath, 'w', encoding='utf-8') as f:
                f.write(rendered_html)
            
            # Step B: Parse with BeautifulSoup
            soup = BeautifulSoup(rendered_html, 'html.parser')
            product_cards = soup.find_all('article', class_='product-card')
            
            # Step C: Loop through expanded cards
            for card in product_cards:
                # Assuming parse_product is defined elsewhere in your script
                parsed_data = parse_product(card)
                
                if parsed_data and parsed_data.get('name'):
                    # Add metadata
                    parsed_data['source_url'] = url
                    parsed_data['scrape_timestamp'] = time.time()
                    parsed_data['category'] = url.split('?')[0].split('/')[-1]
                    page_products.append(parsed_data)
                    
        return page_products

    except Exception as e:
        print(f"Error during extraction from {url}: {e}")
        return []

def fetch_product_data(base_url, path_to_json="migros_products.json", fetch_online=True, verbose=True):
    """
    Main controller: Opens driver, clicks 'Voir plus' until end, then scrapes.
    """
    if not fetch_online:
        if verbose: print(f"Loading local data from {path_to_json}")
        if os.path.exists(path_to_json):
            with open(path_to_json, "r", encoding="utf-8") as f:
                return json.load(f)
        return []

    if verbose:
        print(f"🚀 Starting Chrome to fetch: {base_url}")
    
    driver = create_driver(headless=False)
    all_products = []

    try:
        driver.get(base_url)
        click_count = 0
        
        # ---  DYNAMIC LOADING (Clicking the Button) ---
        while True:
            try:
                # Wait for the button to be present and clickable
                wait = WebDriverWait(driver, 7)
                view_more_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '[data-testid="view-more-button"]')))
                
                # Scroll the button into the center of the screen to avoid click interception
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", view_more_btn)
                time.sleep(1) # Wait for scroll to finish
                
                view_more_btn.click()
                click_count += 1
                
                if verbose:
                    print(f"   [Batch {click_count}] Clicked 'Voir plus'...")
                
                # Wait for new products to inject into the DOM
                time.sleep(2.5) 

            except Exception:
                # This triggers when the button is no longer found (end of category)
                if verbose:
                    print("   ✅ No more 'Voir plus' buttons. Page fully expanded.")
                break

        # --- EXTRACTION ---
        # Pass the SAME driver that now holds all products in its current session
        if verbose:
            print(f"🔍 Extracting data from all loaded products...")
            
        all_products = scrape_single_page(base_url, driver)

    finally:
        driver.quit()

    # ---  SAVE DATA ---
    if all_products:
        with open(path_to_json, "w", encoding="utf-8") as f:
            json.dump(all_products, f, indent=4, ensure_ascii=False)
        if verbose:
            print(f"💾 Successfully saved {len(all_products)} products to '{path_to_json}'.")
    else:
        print("⚠️ No products were extracted.")

    return all_products

In [4]:


def save_to_mongodb(products_list, db_name="migros_db", collection_name="products"):
    """Saves a list of dictionaries to MongoDB, updating existing products."""
    
    # Connect to the local MongoDB server
    client = MongoClient("mongodb://localhost:27017/")
    db = client[db_name]
    collection = db[collection_name]
    
    print(f"Connected to MongoDB. Saving {len(products_list)} products...")
    
    for product in products_list:
        if not product.get('id'):
            continue
    
        collection.insert_one(product)
        
    print("Data successfully saved to MongoDB!")

In [ ]:

for category, url in urls_migros.items():
    if category == "FRUITS_LEGUMES":
    
        print(f"\n--- Scraping category: {category} ---")
        products = fetch_product_data(base_url=BASE_URL + url, path_to_json=f"{category.lower()}.json", fetch_online=True, verbose=True)
        if len(products) == 0:
            print(f"⚠️ No products found for {category}. Skipping MongoDB save.")
            break
        print(f"Total products scraped for {category}: {len(products)}")
        save_to_mongodb(products_list=products, db_name="migros_db", collection_name=collection_name)






--- Scraping category: PRODUITS_LAITIERS_OEUFS_PLATS ---
🚀 Starting Chrome to fetch: https://www.migros.ch/fr/category/produits-laitiers-ufs-plats-prep
   [Batch 1] Clicked 'Voir plus'...
   [Batch 2] Clicked 'Voir plus'...
   [Batch 3] Clicked 'Voir plus'...
   [Batch 4] Clicked 'Voir plus'...
   [Batch 5] Clicked 'Voir plus'...
   [Batch 6] Clicked 'Voir plus'...
   [Batch 7] Clicked 'Voir plus'...
   [Batch 8] Clicked 'Voir plus'...
   [Batch 9] Clicked 'Voir plus'...
   [Batch 10] Clicked 'Voir plus'...
   [Batch 11] Clicked 'Voir plus'...
   [Batch 12] Clicked 'Voir plus'...
   [Batch 13] Clicked 'Voir plus'...
   [Batch 14] Clicked 'Voir plus'...
   [Batch 15] Clicked 'Voir plus'...
   [Batch 16] Clicked 'Voir plus'...
   [Batch 17] Clicked 'Voir plus'...
   [Batch 18] Clicked 'Voir plus'...
   [Batch 19] Clicked 'Voir plus'...
   [Batch 20] Clicked 'Voir plus'...
   [Batch 21] Clicked 'Voir plus'...
   [Batch 22] Clicked 'Voir plus'...
   [Batch 23] Clicked 'Voir plus'...
   [B